# Downloading and processing dataset
Questions with attachments and links are removed.

In [11]:
import os
import re
import shutil
import pandas as pd
from datasets import load_dataset
from huggingface_hub import snapshot_download

local_dir = "./backend/data/gaia"
output_csv = "./backend/data/gaia_validation_annotated.csv"

if os.path.exists(local_dir):
    print("GAIA folder already exists, skipping download...")
    data_dir = local_dir
else:
    print("Downloading GAIA dataset...")
    data_dir = snapshot_download(
        repo_id="gaia-benchmark/GAIA",
        repo_type="dataset",
        local_dir=local_dir
    )
    print("Download complete!")

# Load full validation set (all levels)
print("Loading dataset...")
dataset = load_dataset(data_dir, "2023_all", split="validation")
df = dataset.to_pandas()

# Filter to no-file questions only
df_clean = df[df['file_name']==""].drop(columns=['file_name', 'file_path', 'Annotator Metadata']).reset_index(drop=True)
print(f"Filtered dataset shape: {df_clean.shape}")

# Removing rows with URLs in the "Question" column
url_pattern = re.compile(
    r'(https?://|www\.)\S+|'      # http/https/www links
    r'\b\w+\.(com|org|net|io|gov|edu|co|uk|de|fr)\b',  # bare domain-like patterns
    re.IGNORECASE
)
mask = df_clean["Question"].astype(str).apply(lambda x: bool(url_pattern.search(x)))
df_filtered = df_clean[~mask]

# fetch annotation data and merge with filtered GAIA data
annotations_path = "./backend/data_annotator/annotations.csv"
annotations_df = pd.read_csv(annotations_path)
merged_df = pd.merge(df_filtered, annotations_df, on='task_id', how='inner')


# Save to CSV
os.makedirs("./backend/data", exist_ok=True)
merged_df.to_csv(output_csv, index=False)


print(f"Remaining rows: {len(df_filtered)}")

# Delete the raw GAIA folder (parquet + everything)
shutil.rmtree(local_dir)
print(f"Deleted {local_dir}")

Fetching 119 files: 100%|██████████| 119/119 [00:04<00:00, 25.94it/s]

Download complete!
Loading dataset...
Filtered dataset shape: (127, 4)
Remaining rows: 118
Deleted ./backend/data/gaia


In [10]:
merged_df.head()

,task_id,Question,Level,Final answer,category
0,17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc,I’m researching species that became invasive a...,2,34689,Science & Mathematics
1,04a04a9b-226c-43fd-b319-d5e89743676f,If we assume all articles published by Nature ...,2,41,Science & Mathematics
2,14569e28-c88c-43e4-8c32-097d35b9a67d,"In Unlambda, what exact charcter or text needs...",2,backtick,Technology & Computer Science
3,e1fc63a2-da7a-432f-be78-7c4a95598703,If Eliud Kipchoge could maintain his record-ma...,1,17,Science & Mathematics
4,8e867cd7-cff9-4e6c-867a-ff5ddc2550be,How many studio albums were published by Merce...,1,3,Pop Culture & Entertainment


# Data distribution

In [1]:
import pandas as pd
annotations_path = "./backend/data_annotator/annotations.csv"
df = pd.read_csv(annotations_path)
df['category'].value_counts()

category
Science & Mathematics                   42
Pop Culture & Entertainment             24
Other                                   13
Arts, Literature & Museums              11
Technology & Computer Science            8
Language, Linguistics & Cryptography     8
History & Politics                       6
Geography & Travel                       6
Name: count, dtype: int64

In [5]:
df['Level'].value_counts()

Level
2    60
1    40
3    18
Name: count, dtype: int64

# Generating Question list (.txt file)

In [2]:
input_csv = "./backend/data/gaia_validation_annotated.csv"
output_txt = "./backend/data/gaia_questions_numbered.txt"

df = pd.read_csv(input_csv)

with open(output_txt, "w", encoding="utf-8") as f:
    for i, question in enumerate(df["Question"].values, start=1):
        f.write(f"{i}. {question}\n")

print(f"Saved {len(df)} numbered questions to {output_txt}")

Saved 118 numbered questions to ./backend/data/gaia_questions_numbered.txt
